# MedSAM-VPT — Colab Training

Run this notebook with a GPU runtime (Runtime → Change runtime type → T4 / L4 / A100).

Prerequisites (do once, see `colab/README.md` for details):
1. ISIC 2018 data uploaded to `MyDrive/medsam-vpt-data/{train,val,test}_{images,masks}/`.
2. This repo pushed to GitHub at `github.com/ImSounic/medsam-vpt`.

Edit the `CONFIG` variable below to pick which method to train. Then run cells in order.

In [ ]:
# Which training config to run
CONFIG = "configs/vpt_shallow.yaml"
# Set to True to resume from latest.pth in the run directory if present
RESUME = True
# Where the data lives on Drive
DATA_DRIVE_PATH = "/content/drive/MyDrive/medsam-vpt-data"
# GitHub repo URL
GIT_URL = "https://github.com/ImSounic/medsam-vpt.git"

## 1. Verify GPU

In [ ]:
import torch
assert torch.cuda.is_available(), "GPU not available — change runtime type to GPU."
props = torch.cuda.get_device_properties(0)
print(f"GPU       : {props.name}")
print(f"VRAM      : {props.total_memory / 1e9:.1f} GB")
print(f"PyTorch   : {torch.__version__}")
print(f"CUDA      : {torch.version.cuda}")

## 2. Clone repo (or pull latest)

In [ ]:
import os
REPO_DIR = "/content/medsam-vpt"
if not os.path.exists(REPO_DIR):
    !git clone {GIT_URL} {REPO_DIR}
%cd {REPO_DIR}
!git pull --quiet
!git log -1 --oneline

## 3. Install requirements

Colab already has torch + CUDA so we install the rest.

In [ ]:
!pip install -q -r requirements.txt

## 4. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 5. Symlink data

Points the project's `data/` directory at the Drive folder. Existing
configs (which use `root: data`) keep working unchanged.

In [ ]:
import os
data_dir = f"{REPO_DIR}/data"
if os.path.islink(data_dir) or os.path.exists(data_dir):
    print(f"{data_dir} already exists — skipping symlink")
else:
    os.symlink(DATA_DRIVE_PATH, data_dir)
    print(f"linked {data_dir} -> {DATA_DRIVE_PATH}")
!ls -la data/ | head -10

## 6. Download MedSAM weights

Cached to a Drive subfolder so subsequent sessions skip the download.

In [ ]:
import os
os.makedirs(f"{REPO_DIR}/checkpoints", exist_ok=True)
drive_weights = f"{DATA_DRIVE_PATH}/medsam_vit_b.pth"
local_weights = f"{REPO_DIR}/checkpoints/medsam_vit_b.pth"
if os.path.exists(drive_weights):
    if not os.path.exists(local_weights):
        os.symlink(drive_weights, local_weights)
    print(f"using weights from Drive ({os.path.getsize(local_weights)/1e6:.1f} MB)")
else:
    !python scripts/download_medsam.py
    # Cache to Drive for next time
    !cp {local_weights} {drive_weights}
    print("cached weights to Drive")

## 7. (Optional) Restore previous checkpoints from Drive

If you've trained before and saved to Drive, this brings the run
directories back so you can resume.

In [ ]:
import os
drive_runs = f"{DATA_DRIVE_PATH}/checkpoints_runs"
local_runs = f"{REPO_DIR}/checkpoints/runs"
if os.path.exists(drive_runs):
    os.makedirs(local_runs, exist_ok=True)
    !cp -rn {drive_runs}/* {local_runs}/ 2>/dev/null || true
    !ls {local_runs}
else:
    print("no prior runs on Drive — starting fresh")

## 8. Train

Runs the config you set at the top of the notebook. With `RESUME=True`
and an existing `latest.pth`, training continues from where it left off.

In [ ]:
resume_flag = "--resume" if RESUME else ""
!python -m src.train --config {CONFIG} {resume_flag}

## 9. Save checkpoints back to Drive

**Important** — Colab runtimes get recycled. If you don't push checkpoints
back to Drive, you'll lose them when the session ends.

In [ ]:
import os
os.makedirs(f"{DATA_DRIVE_PATH}/checkpoints_runs", exist_ok=True)
!cp -r checkpoints/runs/* {DATA_DRIVE_PATH}/checkpoints_runs/
print(f"checkpoints synced to {DATA_DRIVE_PATH}/checkpoints_runs")
!ls {DATA_DRIVE_PATH}/checkpoints_runs

## 10. (Optional) Eval the trained checkpoint

Eval is light (~8 min for 1000 images) so you can do it here or pull the
checkpoint to your laptop and run eval there.

In [ ]:
import yaml
with open(CONFIG) as f:
    cfg = yaml.safe_load(f)
run_name = cfg["name"]
ckpt = f"checkpoints/runs/{run_name}/best.pth"
!python -m src.eval --config configs/zero_shot.yaml --checkpoint {ckpt}
# Also sync the updated runs.csv back to Drive
!cp results/runs.csv {DATA_DRIVE_PATH}/runs.csv 2>/dev/null || true